# Cross-tissue/cross-batch label transfer

In [ ]:
mode = "subset"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
colors = light_palette + ["#4c626c", "#003754"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import sys
import os

_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "figS8_S10_neurips2021_bmmc_label_transfer")
os.makedirs(_OUTDIR_ABS, exist_ok=True)

sys.path.append(os.path.abspath("../.experiment_data/primary/benchmarking"))

import contextlib
import functools

_POSTPROC_CWD = os.getcwd()

@contextlib.contextmanager
def _in_primary_dir():
    os.chdir(os.path.join(_POSTPROC_CWD, "../.experiment_data/primary"))
    try:
        yield
    finally:
        os.chdir(_POSTPROC_CWD)

with _in_primary_dir():
    import muon as _muon
    _orig_sample_obs = _muon.pp.sample_obs
    def _patched_sample_obs(data, *a, **kw):
        return _orig_sample_obs(data, *a, **kw).copy()
    _muon.pp.sample_obs = _patched_sample_obs
    import datasets

for _name in dir(datasets):
    if _name.startswith("load_"):
        def _make_wrapper(fn):
            @functools.wraps(fn)
            def _wrapper(*a, **kw):
                with _in_primary_dir():
                    return fn(*a, **kw)
            return _wrapper
        setattr(datasets, _name, _make_wrapper(getattr(datasets, _name)))
import glob
import pandas as pd
import scipy
import numpy as np
import os
from tqdm import tqdm
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import glob
from collections import Counter
from pathlib import Path
from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
import glob
import pandas as pd
import scipy
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from collections import defaultdict
import sklearn
from pysankey import sankey
from sklearn.preprocessing import normalize
import matplotlib.colors as mcolors

os.makedirs("outputs/figS8_S10_neurips2021_bmmc_label_transfer", exist_ok=True)


In [ ]:
for path in sorted(
    glob.glob(
        "../.experiment_data/primary/experiments/*0.6*scarches_*True*/default/label_transfer_metrics_query.pickle"
    )
):
    print(path.split("/")[-3])
    with open(path, "rb") as f:
        data = pickle.load(f)
    print(data["f1_score_micro"])


In [ ]:
for path in sorted(
    glob.glob(
        "../.experiment_data/primary/experiments/*0.6*scarches_*True*/default/label_transfer_metrics_query.pickle"
    )
):
    path = path.replace("0.6", "1.0")
    print(path.split("/")[-3])
    with open(path, "rb") as f:
        data = pickle.load(f)
    print(data["f1_score_micro"])


In [ ]:
query_to_reference_cell_type_mapping = {
    "CD14 Mono": "CD14+ Mono",
    "CD4 Naive": "CD4+ T naive",
    "NK": "NK",
    "CD4 TCM": np.nan,
    "CD8 TEM": np.nan,
    "CD8 Naive": "CD8+ T naive",
    "B naive": "Naive CD20+ B IGKC+",
    "CD16 Mono": "CD16+ Mono",
    "CD4 TEM": "CD4+ T activated",
    "gdT": "gdT TCRVD2+",
    "B memory": np.nan,
    "CD8 TCM": np.nan,
    "MAIT": "MAIT",
    "Treg": "T reg",
    "B intermediate": "Transitional B",
    "Platelet": np.nan,
    "CD4 CTL": "CD4+ T CD314+ CD45RA+",
    "NK_CD56bright": "NK",
    "cDC2": "cDC2",
    "pDC": "pDC",
    "NK Proliferating": "NK",
    "Doublet": np.nan,
    "dnT": "dnT",
    "HSPC": "HSC",
    "ILC": "ILC",
    "Plasmablast": "Plasmablast IGKC+",
    "CD8 Proliferating": np.nan,
    "Eryth": "Erythroblast",
    "ASDC": np.nan,
    "cDC1": "cDC1",
    "CD4 Proliferating": np.nan,
}

ordering_cell_types = [
    "HSC",
    "Lymph prog",
    "G/M prog",
    "MK/E prog",
    "Platelet",
    "ILC",
    "ILC1",
    "NK",
    "NK CD158e1+",
    "CD4+ T naive",
    "CD4 TCM",
    "CD4 Proliferating",
    "CD4+ T activated",
    "CD4+ T activated B7+",
    "CD4+ T CD314+ CD45RA+",
    "CD8+ T naive",
    "CD8+ T naive CD127+ CD26- CD101-",
    "CD8 TCM",
    "T prog cycling",
    "CD8 TEM",
    "CD8 Proliferating",
    "CD8+ T CD49f+",
    "CD8+ T CD69+ CD45RA+",
    "CD8+ T CD69+ CD45RO+",
    "CD8+ T TIGIT+ CD45RA+",
    "CD8+ T TIGIT+ CD45RO+",
    "CD8+ T CD57+ CD45RA+",
    "CD8+ T CD57+ CD45RO+",
    "gdT CD158b+",
    "gdT TCRVD2+",
    "dnT",
    "T reg",
    "Naive CD20+ B IGKC+",
    "Naive CD20+ B IGKC-",
    "Transitional B",
    "B1 B IGKC+",
    "B1 B IGKC-",
    "B memory",
    "Plasma cell IGKC+",
    "Plasma cell IGKC-",
    "Plasmablast IGKC+",
    "Plasmablast IGKC-",
    "cDC1",
    "cDC2",
    "pDC",
    "CD14+ Mono",
    "CD16+ Mono",
    "Doublet",
    "Erythroblast",
    "Normoblast",
    "Proerythroblast",
    "Reticulocyte",
    "MAIT",
    "ASDC",
]


In [ ]:
def map_values(val, return_val=False):
    mapped_value = query_to_reference_cell_type_mapping.get(val, val)
    if mapped_value is np.nan:
        if return_val:
            return val
        else:
            return np.nan
    return mapped_value


In [ ]:
def calculate_metrics(true_labels, predicted_labels, filter_method):
    if filter_method == 'labels':
        true_labels_before_mapping = true_labels.copy()
        vectorized_map = np.vectorize(map_values, otypes=[object])
        true_labels = vectorized_map(true_labels, return_val=True)
    metrics = {'f1_score_micro': sklearn.metrics.f1_score(true_labels, predicted_labels, average='micro')}
    if filter_method == 'count':
        confusion_matrix = sklearn.metrics.confusion_matrix(true_labels, predicted_labels, labels=sorted(np.unique(true_labels)))
    elif filter_method == 'labels':
        confusion_matrix = sklearn.metrics.confusion_matrix(true_labels, predicted_labels, labels=[label for label in ordering_cell_types if label in true_labels or label in predicted_labels])
    if filter_method == 'count':
        unique_labels, counts = np.unique(true_labels, return_counts=True)
        labels_to_keep = unique_labels[counts > 50]
        mask = np.isin(true_labels, labels_to_keep)
    elif filter_method == 'labels':
        true_labels = true_labels_before_mapping.copy()
        vectorized_map = np.vectorize(map_values, otypes=[object])
        true_labels = vectorized_map(true_labels, return_val=False)
        mask = ~np.vectorize(lambda x: isinstance(x, float) and np.isnan(x))(true_labels)
    predicted_labels = predicted_labels[mask]
    true_labels = true_labels[mask]
    metrics_filtered = {'f1_score_micro': sklearn.metrics.f1_score(true_labels, predicted_labels, average='micro')}
    return (metrics, metrics_filtered, confusion_matrix)

def plot_sankey(true_labels, predicted_labels):
    plt.gcf().set_size_inches(10, 15)
    ax1 = plt.axes()
    sankey(true_labels, predicted_labels, leftLabels=sorted(np.unique(true_labels)), rightLabels=sorted(np.unique(predicted_labels)), fontsize=10, ax=ax1)
    plt.show()

def plot_heatmap(confusion_matrix, true_labels, predicted_labels, save_name=None):
    norm_confusion_matrix = normalize(confusion_matrix, axis=0)
    data_norm_confusion_matrix = []
    for true_label_index, true_label in enumerate(sorted(np.unique(true_labels))):
        for pred_label_index, pred_label in enumerate(sorted(np.unique(true_labels))):
            data_norm_confusion_matrix.append([true_label, pred_label, norm_confusion_matrix[true_label_index, pred_label_index]])
    df = pd.DataFrame(columns=['true labels', 'predicted labels', 'prevalence'], data=data_norm_confusion_matrix)
    g = sns.jointplot(data=df, x='true labels', y='predicted labels', kind='hist', bins=(len(sorted(np.unique(true_labels))), len(sorted(np.unique(true_labels)))), cmap=cmap)
    set_figure_params('single', 3.5)
    g.fig.set_size_inches(3.5, 3.5)
    g.ax_joint.grid(False)
    g.ax_marg_x.grid(False)
    g.ax_marg_y.grid(False)
    g.ax_marg_y.cla()
    g.ax_marg_x.cla()
    heatmap = sns.heatmap(data=df['prevalence'].to_numpy().reshape(len(sorted(np.unique(true_labels))), len(sorted(np.unique(true_labels)))), ax=g.ax_joint, cmap=cmap, cbar=False)
    cbar_ax = g.fig.add_axes([1, 0.3, 0.02, 0.4])
    cbar = plt.colorbar(heatmap.get_children()[0], cax=cbar_ax)
    prevalence_true_labels = np.array(list(dict(sorted(Counter(true_labels).items())).values()))
    prevalence_predicted_labels = np.array([dict(sorted(Counter(predicted_labels).items())).get(true_label, 0) for true_label in sorted(np.unique(true_labels))])
    g.ax_marg_y.barh(np.arange(0.5, len(sorted(np.unique(true_labels)))), prevalence_true_labels, color='#003754')
    g.ax_marg_x.bar(np.arange(0.5, len(sorted(np.unique(true_labels)))), prevalence_predicted_labels, color='#003754')
    g.ax_marg_y.set_yticks(np.arange(0.5, len(sorted(np.unique(true_labels)))))
    g.ax_marg_y.set_yticklabels(range(len(sorted(np.unique(true_labels)))))
    step_size = round((prevalence_true_labels.max() - 0) / (5 - 1) / 100) * 100
    max_val = np.ceil(prevalence_true_labels.max() / step_size) * step_size
    g.ax_marg_y.set_xticks(np.arange(0, max_val + step_size, step_size))
    g.ax_marg_y.set_xlabel('prev.')
    g.ax_marg_x.set_xticks(np.arange(0.5, len(sorted(np.unique(true_labels)))))
    g.ax_marg_x.set_xticklabels(range(1, len(sorted(np.unique(true_labels))) + 1), rotation=0)
    step_size = round((prevalence_predicted_labels.max() - 0) / (5 - 1) / 100) * 100
    max_val = np.ceil(prevalence_predicted_labels.max() / step_size) * step_size
    g.ax_marg_x.set_yticks(np.arange(0, max_val + step_size, step_size))
    g.ax_joint.set_xticks(np.arange(0.5, len(sorted(np.unique(true_labels)))))
    g.ax_joint.set_xticklabels(sorted(np.unique(true_labels)), rotation=-90)
    g.ax_joint.set_yticks(np.arange(0.5, len(sorted(np.unique(true_labels)))))
    g.ax_joint.set_yticklabels(sorted(np.unique(true_labels)), rotation=0)
    g.ax_joint.set_xlabel('predicted labels', fontsize=9)
    g.ax_joint.set_ylabel('true labels', fontsize=9)
    g.ax_marg_x.tick_params(axis='x', bottom=False, labelbottom=False)
    g.ax_marg_y.tick_params(axis='y', left=False, labelleft=False)
    g.ax_marg_x.tick_params(axis='y', left=True, labelleft=True)
    g.ax_marg_y.tick_params(axis='x', bottom=True, labelbottom=True, rotation=-90)
    g.ax_marg_x.set_ylabel('prev.', fontsize=10, labelpad=10)
    g.fig.set_size_inches(3.5, 3.5)
    g.fig.subplots_adjust(hspace=0.1, wspace=0.1)
    if save_name is None:
        plt.show()
    else:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')

def plot_heatmap_hao2021(confusion_matrix, true_labels, predicted_labels, mode, save_name=None, grid=False):

    def map_values(val, return_val=False):
        mapped_value = query_to_reference_cell_type_mapping.get(val, val)
        if mapped_value is np.nan:
            if return_val:
                return val
            else:
                return np.nan
        return mapped_value
    true_labels_before_mapping = true_labels.copy()
    vectorized_map = np.vectorize(map_values, otypes=[object])
    true_labels = vectorized_map(true_labels, return_val=True)
    ordering_cell_types_selected = [label for label in ordering_cell_types if label in true_labels or label in predicted_labels]
    selector_true_labels = [True if label in true_labels else False for label in ordering_cell_types_selected]
    selector_predicted_labels = [True if label in predicted_labels else False for label in ordering_cell_types_selected]
    confusion_matrix = confusion_matrix[selector_true_labels, :][:, selector_predicted_labels]
    true_labels_selected = [label for label in ordering_cell_types_selected if label in true_labels]
    predicted_labels_selected = [label for label in ordering_cell_types_selected if label in predicted_labels]
    norm_confusion_matrix = normalize(confusion_matrix, axis=0)
    data_norm_confusion_matrix = []
    for true_label_index, true_label in enumerate(true_labels_selected):
        for pred_label_index, pred_label in enumerate(predicted_labels_selected):
            data_norm_confusion_matrix.append([true_label, pred_label, norm_confusion_matrix[true_label_index, pred_label_index]])
    df = pd.DataFrame(columns=['true labels', 'predicted labels', 'prevalence'], data=data_norm_confusion_matrix)
    aspect_ratio = len(true_labels_selected) / len(predicted_labels_selected)
    colorbar_space = 0.1
    g = sns.jointplot(data=df, x='true labels', y='predicted labels', kind='hist', bins=(len(true_labels_selected), len(predicted_labels_selected)), cmap=cmap)
    set_figure_params('single', 3.1 * aspect_ratio * (1 + colorbar_space))
    if mode == 'subset':
        g.fig.set_size_inches(3.5, 3.4 * aspect_ratio * (1 + colorbar_space))
    elif mode == 'all':
        g.fig.set_size_inches(3.5, 2.7)
    else:
        raise NotImplementedError('')
    g.ax_joint.grid(False)
    g.ax_marg_x.grid(False)
    g.ax_marg_y.grid(False)
    g.ax_marg_y.cla()
    g.ax_marg_x.cla()
    heatmap = sns.heatmap(data=df['prevalence'].to_numpy().reshape(len(true_labels_selected), len(predicted_labels_selected)), ax=g.ax_joint, cmap=cmap, cbar=False)
    if grid:
        n_rows, n_cols = df['prevalence'].to_numpy().reshape(len(true_labels_selected), len(predicted_labels_selected)).shape
        for i in range(n_rows):
            for j in range(n_cols):
                heatmap.hlines(i + 0.5, j, j + 1, color='black', linewidth=0.2)
                heatmap.vlines(j + 0.5, i, i + 1, color='black', linewidth=0.2)
    cbar_ax = g.fig.add_axes([1, 0.3, 0.02, 0.4])
    cbar = plt.colorbar(heatmap.get_children()[0], cax=cbar_ax)
    prevalence_true_labels = np.array([Counter(true_labels)[label] for label in true_labels_selected])
    prevalence_predicted_labels = np.array([Counter(predicted_labels)[label] for label in predicted_labels_selected])
    g.ax_marg_y.barh(np.arange(0.5, len(prevalence_true_labels)), prevalence_true_labels, color='#003754')
    g.ax_marg_x.bar(np.arange(0.5, len(prevalence_predicted_labels)), prevalence_predicted_labels, color='#003754')
    g.ax_marg_y.set_yticks(np.arange(0.5, len(prevalence_true_labels)))
    g.ax_marg_y.set_yticklabels(range(len(prevalence_true_labels)))
    step_size = round((prevalence_true_labels.max() - 0) / (5 - 1) / 100) * 100
    max_val = np.ceil(prevalence_true_labels.max() / step_size) * step_size
    g.ax_marg_y.set_xticks(np.arange(0, max_val + step_size, step_size))
    g.ax_marg_y.set_xlabel('prevalence')
    g.ax_marg_x.set_xticks(np.arange(0.5, len(predicted_labels_selected)))
    g.ax_marg_x.set_xticklabels(range(1, len(predicted_labels_selected) + 1), rotation=0)
    step_size = round((prevalence_predicted_labels.max() - 0) / (3 - 1) / 100) * 100
    max_val = np.ceil(prevalence_predicted_labels.max() / step_size) * step_size
    g.ax_marg_x.set_yticks(np.arange(0, max_val + step_size, step_size))
    g.ax_marg_x.set_ylabel('prevalence')
    g.ax_joint.set_xticks(np.arange(0.5, len(predicted_labels_selected)))
    g.ax_joint.set_xticklabels(predicted_labels_selected, rotation=-90)
    g.ax_joint.set_yticks(np.arange(0.5, len(true_labels_selected)))
    g.ax_joint.set_yticklabels(true_labels_selected, rotation=0)
    g.ax_joint.set_xlabel('predicted labels')
    g.ax_joint.set_ylabel('true labels')
    g.ax_marg_x.tick_params(axis='x', bottom=False, labelbottom=False)
    g.ax_marg_y.tick_params(axis='y', left=False, labelleft=False)
    g.ax_marg_x.tick_params(axis='y', left=True, labelleft=True)
    g.ax_marg_y.tick_params(axis='x', bottom=True, labelbottom=True, rotation=-90)
    g.fig.subplots_adjust(hspace=0.1, wspace=0.1)
    if save_name is None:
        plt.show()
    else:
        print(save_name)
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')

def plot_metric(mode, metric_values, metric_label, save_name):

    def lighten_color(color, amount=0.6):
        c = np.array(mcolors.to_rgba(color))
        white = np.array([1, 1, 1, 1])
        return tuple((1 - amount) * c + amount * white)
    base_colors = plt.rcParams['axes.prop_cycle'].by_key()['color'][:5]
    light_colors = [lighten_color(color, amount=0.4) for color in base_colors]
    if mode == 'all':
        set_figure_params('single', 2)
    else:
        set_figure_params('double', 2.5)
    fig, ax = plt.subplots()
    indices = np.arange(len(['MultiVI', 'SparseVI', 'MultiVI', 'SparseVI', 'MultiVI', 'SparseVI']))
    max_height = 0
    bar_width = 0.2
    for i, group_data in enumerate(metric_values):
        num_bars = group_data.shape[0]
        for j in range(3):
            bottom = np.zeros(group_data.shape[1])
            for k in range(group_data.shape[1]):
                if j == 2 and (i == 2 or i == 3):
                    color_j = 3
                else:
                    color_j = j
                color = base_colors[color_j] if k != 1 else light_colors[color_j]
                x_pos = indices[i] - bar_width * (num_bars / 2 - 0.5) + j * bar_width
                ax.bar(x_pos, group_data[j, k], bar_width, bottom=bottom, color=color, capsize=3, edgecolor='black')
                bottom += group_data[j, k]
            max_height = max(max_height, bottom.max())
    ax.set_xticks(list(range(6)))
    if mode == 'all':
        ax.set_xticklabels(['MultiVI', 'SparseVI', 'MultiVI', 'SparseVI', 'MultiVI', 'SparseVI'])
    else:
        ax.set_xticklabels(['MultiVI', 'SparseVI', 'MultiVI', 'SparseVI', 'MultiVI', 'SparseVI'])
    ax.set_ylabel(metric_label)
    ax.set_ylim(bottom=0, top=max_height * 1.1)
    legend_handles_tools = [plt.Rectangle((0, 0), 1, 1, color=color, edgecolor='black') for color in base_colors]
    ax.axvline(x=1.5, ymin=0.02, ymax=0.98, color='black', linestyle='dashed', linewidth=1.0)
    ax.axvline(x=3.5, ymin=0.02, ymax=0.98, color='black', linestyle='dashed', linewidth=1.0)
    tool_names = ['Both modalities', 'RNA only', 'Protein only', 'ATAC only']
    ax.legend(legend_handles_tools, tool_names, loc='upper left', bbox_to_anchor=(1.05, 1))
    xlims = ax.get_xlim()
    plot_width = xlims[1] - xlims[0]
    if mode == 'all':
        y_offset = -0.18 * (ax.get_ylim()[1] - ax.get_ylim()[0])
        ax.text(xlims[0] + plot_width / 6, y_offset, 'NeurIPS 2021 CITE', va='center', ha='center', fontsize=4, weight='bold')
        ax.text(xlims[0] + 3 * plot_width / 6, y_offset, 'NeurIPS 2021 Multiome', va='center', ha='center', fontsize=4, weight='bold')
        ax.text(xlims[0] + 5 * plot_width / 6, y_offset, 'NeurIPS 2021 CITE\n-> Hao2021 CITE', va='center', ha='center', fontsize=4, weight='bold')
    else:
        y_offset = -0.32 * (ax.get_ylim()[1] - ax.get_ylim()[0])
        ax.text(xlims[0] + plot_width / 6, y_offset, 'NeurIPS 2021 CITE', va='center', ha='center', fontsize=10.5, weight='bold')
        ax.text(xlims[0] + 3 * plot_width / 6, y_offset, 'NeurIPS 2021 Multiome', va='center', ha='center', fontsize=10.5, weight='bold')
        ax.text(xlims[0] + 5 * plot_width / 6, y_offset, 'NeurIPS 2021 CITE\n-> Hao2021 CITE', va='center', ha='center', fontsize=10.5, weight='bold')
    plt.tight_layout()
    if save_name is None:
        plt.show()
    else:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')

def plot_metric_vertical_old(mode, metric_values, metric_label, bootstrap_upper_metric_values, bootstrap_lower_metric_values, save_name):

    def lighten_color(color, amount=0.6):
        c = np.array(mcolors.to_rgba(color))
        white = np.array([1, 1, 1, 1])
        return tuple((1 - amount) * c + amount * white)
    base_colors = plt.rcParams['axes.prop_cycle'].by_key()['color'][:5]
    light_colors = [lighten_color(color, amount=0.4) for color in base_colors]
    if mode == 'all':
        set_figure_params('single', 2)
    else:
        set_figure_params('single', 2)
    fig, ax = plt.subplots()
    indices = np.arange(len(['MultiVI', 'SparseVI', 'MultiVI', 'SparseVI', 'MultiVI', 'SparseVI']))
    indices = indices[::-1]
    max_width = 0
    bar_height = 0.2
    for i, (group_data, bootstrap_upper_group_data, bootstrap_lower_group_data) in enumerate(zip(metric_values[::-1], bootstrap_upper_metric_values[::-1], bootstrap_lower_metric_values[::-1])):
        num_bars = group_data.shape[0]
        for j in range(3):
            left = np.zeros(group_data.shape[1])
            for k in range(group_data.shape[1]):
                if j == 2 and (i == 2 or i == 3):
                    color_j = 3
                else:
                    color_j = j
                color = base_colors[color_j] if k != 1 else light_colors[color_j]
                y_pos = indices[i] - bar_height * (num_bars / 2 - 0.5) + j * bar_height
                if group_data[j, k] > 0 and bootstrap_lower_group_data[j, k] > 0 and (bootstrap_upper_group_data[j, k] > 0):
                    print(group_data[j, k] - bootstrap_lower_group_data[j, k])
                    print(bootstrap_upper_group_data[j, k] - group_data[j, k])
                    ax.barh(y_pos, group_data[j, k], bar_height, xerr=[group_data[j, k] - bootstrap_lower_group_data[j, k], bootstrap_upper_group_data[j, k] - group_data[j, k]], left=left, color=color, capsize=1, edgecolor='black')
                else:
                    ax.barh(y_pos, group_data[j, k], bar_height, left=left, color=color, capsize=1, edgecolor='black')
                left += group_data[j, k]
            max_width = max(max_width, left.max())
        ax.axhline(y=1.5, xmin=0.02, xmax=0.98, color='black', linestyle='dashed', linewidth=1.0)
    ax.axhline(y=3.5, xmin=0.02, xmax=0.98, color='black', linestyle='dashed', linewidth=1.0)
    ax.set_yticks(indices)
    if mode == 'all':
        ax.set_yticklabels(['MultiVI', 'SparseVI', 'MultiVI', 'SparseVI', 'MultiVI', 'SparseVI'][::-1])
    else:
        ax.set_yticklabels(['MultiVI', 'SparseVI', 'MultiVI', 'SparseVI', 'MultiVI', 'SparseVI'][::-1])
    ax.set_xlabel(metric_label)
    ax.set_xlim(left=0, right=max_width * 1.1)
    tool_names = ['Both modalities', 'RNA only', 'Protein only', 'ATAC only']
    legend_handles_tools = [plt.Rectangle((0, 0), 1, 1, color=color, edgecolor='black') for color in base_colors]
    ax.legend(legend_handles_tools, tool_names, loc='upper center', bbox_to_anchor=(0.3, 1.15), ncol=len(tool_names), frameon=False)
    ylims = ax.get_ylim()
    plot_height = ylims[1] - ylims[0]
    if mode == 'all':
        x_offset = -0.18 * (ax.get_xlim()[1] - ax.get_xlim()[0])
        ax.text(x_offset, ylims[0] + plot_height / 6, 'NeurIPS 2021\nMultiome', va='center', ha='center', rotation=90, fontsize=3.5, weight='bold')
        ax.text(x_offset, ylims[0] + 3 * plot_height / 6, 'NeurIPS 2021\nCITE', va='center', ha='center', rotation=90, fontsize=3.5, weight='bold')
        ax.text(x_offset, ylims[0] + 5 * plot_height / 6, 'NeurIPS 2021 CITE\n-> Hao2021 CITE', va='center', ha='center', rotation=90, fontsize=3.5, weight='bold')
    else:
        x_offset = -0.32 * (ax.get_xlim()[1] - ax.get_xlim()[0])
        ax.text(x_offset, ylims[0] + plot_height / 6, 'NeurIPS\n2021\nMultiome', va='center', ha='center', rotation=90, fontsize=5.5, weight='bold')
        ax.text(x_offset, ylims[0] + 3 * plot_height / 6, 'NeurIPS\n2021\nCITE', va='center', ha='center', rotation=90, fontsize=5.5, weight='bold')
        ax.text(x_offset, ylims[0] + 5 * plot_height / 6, 'NeurIPS\n2021CITE\n-> Hao2021\nCITE', va='center', ha='center', rotation=90, fontsize=5.5, weight='bold')
    plt.tight_layout()
    if save_name is None:
        plt.show()
    else:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')

def plot_metric_vertical(mode, metric_values, metric_label, bootstrap_upper_metric_values, bootstrap_lower_metric_values, save_name):

    def lighten_color(color, amount=0.6):
        c = np.array(mcolors.to_rgba(color))
        white = np.array([1, 1, 1, 1])
        return tuple((1 - amount) * c + amount * white)
    base_colors = plt.rcParams['axes.prop_cycle'].by_key()['color'][:5]
    light_colors = [lighten_color(color, amount=0.4) for color in base_colors]
    if mode == 'all':
        set_figure_params(2.5, 2)
    else:
        set_figure_params(2.6, 2)
    fig, ax = plt.subplots()
    indices = np.arange(len(['MultiVI', 'SparseVI', 'MultiVI', 'SparseVI', 'MultiVI', 'SparseVI']))
    indices = indices[::-1]
    max_width = 0
    bar_height = 0.2
    for i, (group_data, bootstrap_upper_group_data, bootstrap_lower_group_data) in enumerate(zip(metric_values[::-1], bootstrap_upper_metric_values[::-1], bootstrap_lower_metric_values[::-1])):
        num_bars = group_data.shape[0]
        for j in range(3):
            left = np.zeros(group_data.shape[1])
            for k in range(group_data.shape[1]):
                if j == 2 and (i == 2 or i == 3):
                    color_j = 3
                else:
                    color_j = j
                color = base_colors[color_j] if k != 1 else light_colors[color_j]
                y_pos = indices[i] - bar_height * (num_bars / 2 - 0.5) + j * bar_height
                if group_data[j, k] > 0 and bootstrap_lower_group_data[j, k] > 0 and (bootstrap_upper_group_data[j, k] > 0):
                    print(group_data[j, k] - bootstrap_lower_group_data[j, k])
                    print(bootstrap_upper_group_data[j, k] - group_data[j, k])
                    ax.barh(y_pos, group_data[j, k], bar_height, xerr=[group_data[j, k] - bootstrap_lower_group_data[j, k], bootstrap_upper_group_data[j, k] - group_data[j, k]], left=left, color=color, capsize=1, edgecolor='black')
                else:
                    ax.barh(y_pos, group_data[j, k], bar_height, left=left, color=color, capsize=1, edgecolor='black')
                left += group_data[j, k]
            max_width = max(max_width, left.max())
        ax.axhline(y=1.5, xmin=0.02, xmax=0.98, color='black', linestyle='dashed', linewidth=1.0)
    ax.axhline(y=3.5, xmin=0.02, xmax=0.98, color='black', linestyle='dashed', linewidth=1.0)
    ax.set_yticks(indices)
    if mode == 'all':
        ax.set_yticklabels(['MultiVI', 'SparseVI', 'MultiVI', 'SparseVI', 'MultiVI', 'SparseVI'][::-1])
    else:
        ax.set_yticklabels(['MultiVI', 'SparseVI', 'MultiVI', 'SparseVI', 'MultiVI', 'SparseVI'][::-1])
    ax.set_xlabel(metric_label)
    ax.set_xlim(left=0, right=max_width * 1.1)
    tool_names = ['Both modalities', 'RNA only', 'Protein only', 'ATAC only']
    legend_handles_tools = [plt.Rectangle((0, 0), 1, 1, color=color, edgecolor='black') for color in base_colors]
    ax.legend(legend_handles_tools, tool_names, loc='upper center', bbox_to_anchor=(0.3, 1.15), ncol=len(tool_names), frameon=False)
    ylims = ax.get_ylim()
    plot_height = ylims[1] - ylims[0]
    if mode == 'all':
        x_offset = -0.5 * (ax.get_xlim()[1] - ax.get_xlim()[0])
        ax.text(x_offset, ylims[0] + plot_height / 6, 'NeurIPS 2021\nMultiome', va='center', ha='center', rotation=90, fontsize=3.5, weight='bold')
        ax.text(x_offset, ylims[0] + 3 * plot_height / 6, 'NeurIPS 2021\nCITE', va='center', ha='center', rotation=90, fontsize=3.5, weight='bold')
        ax.text(x_offset, ylims[0] + 5 * plot_height / 6, 'NeurIPS 2021 CITE\n-> Hao2021 CITE', va='center', ha='center', rotation=90, fontsize=3.5, weight='bold')
    else:
        x_offset = -0.5 * (ax.get_xlim()[1] - ax.get_xlim()[0])
        ax.text(x_offset, ylims[0] + plot_height / 6, 'NeurIPS\n2021\nMultiome', va='center', ha='center', rotation=90, fontsize=5.5, weight='bold')
        ax.text(x_offset, ylims[0] + 3 * plot_height / 6, 'NeurIPS\n2021\nCITE', va='center', ha='center', rotation=90, fontsize=5.5, weight='bold')
        ax.text(x_offset, ylims[0] + 5 * plot_height / 6, 'NeurIPS\n2021CITE\n-> Hao2021\nCITE', va='center', ha='center', rotation=90, fontsize=5.5, weight='bold')
    plt.tight_layout()
    if save_name is None:
        plt.show()
    else:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')


In [ ]:
def show_umap_plot(X_umap, labels, continuous=False, title=""):

    if not continuous:
        unique_labels, encoded_labels = np.unique(labels, return_inverse=True)
        n_categories = len(unique_labels)
        cmap = ListedColormap(sns.color_palette("husl", n_categories))
    else:
        encoded_labels = labels

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect("equal", "box")

    if continuous:
        sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], s=0.01, cmap=cmap, c=encoded_labels)
    else:
        sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], s=0.01, c=encoded_labels, cmap=cmap)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_box_aspect(1)

    plt.xlabel("UMAP 1", fontsize=12)
    plt.ylabel("UMAP 2", fontsize=12)
    plt.title(title)
    plt.show()

    plt.close()

def show_colorbar():
    fig, ax = plt.subplots(figsize=(0.5, 1))
    sm = plt.cm.ScalarMappable(
        cmap=sns.light_palette("#70acc0", as_cmap=True), norm=plt.Normalize(vmin=0, vmax=1)
    )
    cbar = plt.colorbar(sm, ax=ax)
    cbar.ax.tick_params(labelsize=10)
    cbar.set_label("GO activation", rotation=270, labelpad=15)
    plt.axis("off")
    plt.show()

def show_legend(labels):
    unique_labels, encoded_labels = np.unique(labels, return_inverse=True)
    n_categories = len(unique_labels)
    cmap = ListedColormap(sns.color_palette("husl", n_categories))

    fig, ax = plt.subplots(figsize=(0.5, 1))
    handles = [
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=cmap(i), markersize=10)
        for i in range(n_categories)
    ]
    ax.legend(handles, unique_labels, bbox_to_anchor=(1, 1), loc="upper left")
    plt.axis("off")
    plt.show()


In [ ]:
save_names = [
    f"neurips2021_multiome_bmmc_label_transfer_all_MultiVI_{mode}",
    f"neurips2021_multiome_bmmc_label_transfer_rna_MultiVI_{mode}",
    f"neurips2021_multiome_bmmc_label_transfer_atac_MultiVI_{mode}",
    f"neurips2021_multiome_bmmc_label_transfer_all_SparseVI_{mode}",
    f"neurips2021_multiome_bmmc_label_transfer_rna_SparseVI_{mode}",
    f"neurips2021_multiome_bmmc_label_transfer_atac_SparseVI_{mode}",
    f"neurips2021_cite_bmmc_label_transfer_all_MultiVI_{mode}",
    f"neurips2021_cite_bmmc_label_transfer_rna_MultiVI_{mode}",
    f"neurips2021_cite_bmmc_label_transfer_prot_MultiVI_{mode}",
    f"neurips2021_cite_bmmc_label_transfer_all_SparseVI_{mode}",
    f"neurips2021_cite_bmmc_label_transfer_rna_SparseVI_{mode}",
    f"neurips2021_cite_bmmc_label_transfer_prot_SparseVI_{mode}",
    f"neurips2021_cite_bmmc_hao2021_label_transfer_all_MultiVI_{mode}",
    f"neurips2021_cite_bmmc_hao2021_label_transfer_rna_MultiVI_{mode}",
    f"neurips2021_cite_bmmc_hao2021_label_transfer_prot_MultiVI_{mode}",
    f"neurips2021_cite_bmmc_hao2021_label_transfer_all_SparseVI_{mode}",
    f"neurips2021_cite_bmmc_hao2021_label_transfer_rna_SparseVI_{mode}",
    f"neurips2021_cite_bmmc_hao2021_label_transfer_prot_SparseVI_{mode}",
]


In [ ]:
filter_methods = [
    "count",
    "count",
    "count",
    "count",
    "count",
    "count",
    "count",
    "count",
    "count",
    "count",
    "count",
    "count",
    "labels",
    "labels",
    "labels",
    "labels",
    "labels",
    "labels",
]


In [ ]:
paths = [
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_all_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_rna_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_atac_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_all_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_rna_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_atac_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_all_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_rna_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_prot_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_all_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_rna_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_prot_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_all_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_rna_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_prot_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_all_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_rna_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_prot_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
]


In [ ]:
paths_hyper = [
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_all_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_rna_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_atac_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_all_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_rna_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_multiome_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_atac_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    str(
        Path(
            glob.glob(
                os.path.join(
                    "/".join(paths[6].split("/")[:-2]), "hyper", "hyper*", "labels_query.npy"
                )
            )[0]
        ).parent
    ),
    str(
        Path(
            glob.glob(
                os.path.join(
                    "/".join(paths[7].split("/")[:-2]), "hyper", "hyper*", "labels_query.npy"
                )
            )[0]
        ).parent
    ),
    str(
        Path(
            glob.glob(
                os.path.join(
                    "/".join(paths[8].split("/")[:-2]), "hyper", "hyper*", "labels_query.npy"
                )
            )[0]
        ).parent
    ),
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_all_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_rna_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_prot_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_all_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_rna_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_prot_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_all_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_rna_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_prot_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/",
]


In [ ]:
metrics_labels = [
    "f1 score (micro)",
    "f1 score (macro)",
    "f1 score (weighted)",
    "accuracy score",
    "balanced accuracy score",
    "precision score (micro)",
    "precision score (macro)",
    "precision score (weighted)",
    "recall score (micro)",
    "recall score (macro)",
    "recall score (weighted)",
]


In [ ]:
N_BOOTSTRAPS = 200

def main(mode, paths, filter_methods, save_names):
    metrics_aggregated = defaultdict(list)
    metrics_filtered_aggregated = defaultdict(list)
    bootstrap_mean_metrics_aggregated = defaultdict(list)
    bootstrap_mean_metrics_filtered_aggregated = defaultdict(list)
    bootstrap_ci_lower_metrics_aggregated = defaultdict(list)
    bootstrap_ci_lower_metrics_filtered_aggregated = defaultdict(list)
    bootstrap_ci_higher_metrics_aggregated = defaultdict(list)
    bootstrap_ci_higher_metrics_filtered_aggregated = defaultdict(list)

    for path, filter_method, save_name in tqdm(zip(paths, filter_methods, save_names)):

        print(path)

        true_labels = np.load(os.path.join(path, "labels_query.npy"), allow_pickle=True)
        predicted_labels = np.load(os.path.join(path, "pred_query.npy"), allow_pickle=True)
        true_labels[true_labels == "CD4+ T activated integrinB7+"] = "CD4+ T activated B7+"
        predicted_labels[predicted_labels == "CD4+ T activated integrinB7+"] = (
            "CD4+ T activated B7+"
        )

        metrics, metrics_filtered, confusion_matrix = calculate_metrics(
            true_labels, predicted_labels, filter_method
        )

        for metric in metrics:
            metrics_aggregated[metric].append(metrics[metric])
            metrics_filtered_aggregated[metric].append(metrics_filtered[metric])

        bootstrap_true_labels_original = true_labels.copy()
        bootstrap_predicted_labels_original = predicted_labels.copy()

        if mode != "all":
            true_labels_counts = np.unique(true_labels, return_counts=True)
            predicted_labels_count = np.unique(predicted_labels, return_counts=True)
            true_labels_unique_filt = true_labels_counts[0][true_labels_counts[1] > 50]
            predicted_labels_unique_filt = predicted_labels_count[0][
                predicted_labels_count[1] > 50
            ]
            true_predicted_labels_filt = np.union1d(
                true_labels_unique_filt, predicted_labels_unique_filt
            )
            true_labels_filt = [
                true_label for true_label in true_labels if true_label in true_predicted_labels_filt
            ]
            predicted_labels_filt = [
                predicted_label
                for predicted_label in predicted_labels
                if predicted_label in true_predicted_labels_filt
            ]
            indices_confusion_matrix = [
                True if label in true_predicted_labels_filt else False
                for label in sorted(np.unique(true_labels))
            ]
            if filter_method == "count":
                true_labels_filt = [
                    true_label
                    for true_label in true_labels
                    if true_label in true_predicted_labels_filt
                ]
                predicted_labels_filt = [
                    predicted_label
                    for predicted_label in predicted_labels
                    if predicted_label in true_predicted_labels_filt
                ]
                indices_confusion_matrix = [
                    True if label in true_predicted_labels_filt else False
                    for label in sorted(np.unique(true_labels))
                ]
            elif filter_method == "labels":
                true_labels_filt = [
                    true_label
                    for true_label in true_labels
                    if true_label in true_predicted_labels_filt
                    and true_label in ordering_cell_types
                ]
                predicted_labels_filt = [
                    predicted_label
                    for predicted_label in predicted_labels
                    if predicted_label in true_predicted_labels_filt
                    and predicted_label in ordering_cell_types
                ]
                vectorized_map = np.vectorize(map_values, otypes=[object])
                true_labels_temp = vectorized_map(true_labels, return_val=True)
                indices_confusion_matrix = [
                    True if label in true_predicted_labels_filt else False
                    for label in [
                        label
                        for label in ordering_cell_types
                        if label in true_labels_temp or label in predicted_labels
                    ]
                ]
            confusion_matrix_filt = confusion_matrix[indices_confusion_matrix, :][
                :, indices_confusion_matrix
            ]
        else:
            true_labels_filt = true_labels
            predicted_labels_filt = predicted_labels
            confusion_matrix_filt = confusion_matrix

        bootstrap_metrics_all = defaultdict(list)
        bootstrap_metrics_filtered_all = defaultdict(list)
        for _ in range(N_BOOTSTRAPS):
            bootstrap_indices = np.random.choice(
                len(bootstrap_true_labels_original),
                size=len(bootstrap_true_labels_original),
                replace=True,
            )
            bootstrap_true_labels = bootstrap_true_labels_original[bootstrap_indices]
            bootstrap_predicted_labels = bootstrap_predicted_labels_original[bootstrap_indices]
            bootstrap_metrics, bootstrap_metrics_filtered, confusion_matrix = calculate_metrics(
                bootstrap_true_labels, bootstrap_predicted_labels, filter_method
            )
            for bootstrap_metric in bootstrap_metrics:
                bootstrap_metrics_all[bootstrap_metric].append(bootstrap_metrics[bootstrap_metric])
                bootstrap_metrics_filtered_all[bootstrap_metric].append(
                    bootstrap_metrics_filtered[bootstrap_metric]
                )

        for bootstrap_metric in bootstrap_metrics:
            bootstrap_mean_metrics_aggregated[bootstrap_metric].append(
                np.mean(bootstrap_metrics_all[bootstrap_metric])
            )
            bootstrap_mean_metrics_filtered_aggregated[bootstrap_metric].append(
                np.mean(bootstrap_metrics_filtered_all[bootstrap_metric])
            )
            bootstrap_ci_lower_metrics_aggregated[bootstrap_metric].append(
                np.percentile(bootstrap_metrics_all[bootstrap_metric], 2.5)
            )
            bootstrap_ci_lower_metrics_filtered_aggregated[bootstrap_metric].append(
                np.percentile(bootstrap_metrics_filtered_all[bootstrap_metric], 2.5)
            )
            bootstrap_ci_higher_metrics_aggregated[bootstrap_metric].append(
                np.percentile(bootstrap_metrics_all[bootstrap_metric], 97.5)
            )
            bootstrap_ci_higher_metrics_filtered_aggregated[bootstrap_metric].append(
                np.percentile(bootstrap_metrics_filtered_all[bootstrap_metric], 97.5)
            )

    return (
        metrics_aggregated,
        metrics_filtered_aggregated,
        bootstrap_mean_metrics_aggregated,
        bootstrap_mean_metrics_filtered_aggregated,
        bootstrap_ci_lower_metrics_aggregated,
        bootstrap_ci_higher_metrics_aggregated,
        bootstrap_ci_lower_metrics_filtered_aggregated,
        bootstrap_ci_higher_metrics_filtered_aggregated,
    )


In [ ]:
path = "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_all_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/"
save_name = "neurips2021_cite_bmmc_hao2021_label_transfer_all_SparseVI_full_subset"


In [ ]:
true_labels = np.load(os.path.join(path, "labels_query.npy"), allow_pickle=True)
predicted_labels = np.load(os.path.join(path, "pred_query.npy"), allow_pickle=True)
true_labels[true_labels == "CD4+ T activated integrinB7+"] = "CD4+ T activated B7+"
predicted_labels[predicted_labels == "CD4+ T activated integrinB7+"] = "CD4+ T activated B7+"

metrics, metrics_filtered, confusion_matrix = calculate_metrics(
    true_labels, predicted_labels, "labels"
)

true_labels_filt = true_labels
predicted_labels_filt = predicted_labels
confusion_matrix_filt = confusion_matrix
plot_heatmap_hao2021(
    confusion_matrix_filt, true_labels_filt, predicted_labels_filt, mode, save_name, grid=True
)


In [ ]:
required_indices_count = [0, 3, 6, 9]
required_indices_labels = [12, 13, 14, 16, 17]

for idx in required_indices_count:
    path = paths[idx]
    save_name = save_names[idx]
    true_labels = np.load(os.path.join(path, "labels_query.npy"), allow_pickle=True)
    predicted_labels = np.load(os.path.join(path, "pred_query.npy"), allow_pickle=True)
    true_labels[true_labels == "CD4+ T activated integrinB7+"] = "CD4+ T activated B7+"
    predicted_labels[predicted_labels == "CD4+ T activated integrinB7+"] = "CD4+ T activated B7+"
    metrics, metrics_filtered, confusion_matrix = calculate_metrics(
        true_labels, predicted_labels, "count"
    )
    plot_heatmap(confusion_matrix, true_labels, predicted_labels, save_name=save_name)

for idx in required_indices_labels:
    path = paths[idx]
    save_name = save_names[idx]
    true_labels = np.load(os.path.join(path, "labels_query.npy"), allow_pickle=True)
    predicted_labels = np.load(os.path.join(path, "pred_query.npy"), allow_pickle=True)
    true_labels[true_labels == "CD4+ T activated integrinB7+"] = "CD4+ T activated B7+"
    predicted_labels[predicted_labels == "CD4+ T activated integrinB7+"] = "CD4+ T activated B7+"
    metrics, metrics_filtered, confusion_matrix = calculate_metrics(
        true_labels, predicted_labels, "labels"
    )
    plot_heatmap_hao2021(
        confusion_matrix, true_labels, predicted_labels, mode, save_name, grid=True
    )


In [ ]:
(
    metrics_aggregated,
    metrics_filtered_aggregated,
    bootstrap_mean_metrics_aggregated,
    bootstrap_mean_metrics_filtered_aggregated,
    bootstrap_ci_lower_metrics_aggregated,
    bootstrap_ci_higher_metrics_aggregated,
    bootstrap_ci_lower_metrics_filtered_aggregated,
    bootstrap_ci_higher_metrics_filtered_aggregated,
) = main(mode, paths, filter_methods, save_names)


In [ ]:
mode = "subset"


In [ ]:
(
    metrics_aggregated,
    metrics_filtered_aggregated,
    bootstrap_mean_metrics_aggregated,
    bootstrap_mean_metrics_filtered_aggregated,
    bootstrap_ci_lower_metrics_aggregated,
    bootstrap_ci_higher_metrics_aggregated,
    bootstrap_ci_lower_metrics_filtered_aggregated,
    bootstrap_ci_higher_metrics_filtered_aggregated,
) = main(mode, paths, filter_methods, save_names)


In [ ]:
mode = "all"


In [ ]:
(
    metrics_aggregated,
    metrics_filtered_aggregated,
    bootstrap_mean_metrics_aggregated,
    bootstrap_mean_metrics_filtered_aggregated,
    bootstrap_ci_lower_metrics_aggregated,
    bootstrap_ci_higher_metrics_aggregated,
    bootstrap_ci_lower_metrics_filtered_aggregated,
    bootstrap_ci_higher_metrics_filtered_aggregated,
) = main(mode, paths, filter_methods, save_names)


In [ ]:
import pickle

paths = [
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_all_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/label_transfer_metrics_query.pickle",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_rna_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/label_transfer_metrics_query.pickle",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_prot_ec_True_ufl_True_batcov_True_patcov_False_modtype_dense_s0/default/label_transfer_metrics_query.pickle",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_all_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/label_transfer_metrics_query.pickle",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_rna_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/label_transfer_metrics_query.pickle",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_hao_query_True_mode_prot_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/label_transfer_metrics_query.pickle",
]
for path in paths:
    with open(path, "rb") as f:
        data = pickle.load(f)
    print(data["f1_score_micro"])


In [ ]:
import pickle

paths = [
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_all_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/label_transfer_metrics_query.pickle",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_rna_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/label_transfer_metrics_query.pickle",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_False_patscib_False_scarches_True_mode_prot_ec_True_dic_True_ufl_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/label_transfer_metrics_query.pickle",
]
for path in paths:
    with open(path, "rb") as f:
        data = pickle.load(f)
    print(data["f1_score_micro"])


In [ ]:
def plot_metric_vertical_v2(mode, metric_values, metric_label, bootstrap_upper_metric_values, bootstrap_lower_metric_values, save_name):
    groups = 9
    categories = ['SparseVI', 'MultiVI']
    values_multivi = np.concatenate([metric_values.T[0][:3][::-1], metric_values.T[0][6:9][::-1], metric_values.T[0][12:15][::-1]])
    values_sparsevi = np.concatenate([metric_values.T[0][3:6][::-1], metric_values.T[0][9:12][::-1], metric_values.T[0][15:][::-1]])
    bootstrap_upper_metric_values_multivi = np.concatenate([bootstrap_upper_metric_values.T[0][:3][::-1], bootstrap_upper_metric_values.T[0][6:9][::-1], bootstrap_upper_metric_values.T[0][12:15][::-1]])
    bootstrap_lower_metric_values_multivi = np.concatenate([bootstrap_lower_metric_values.T[0][:3][::-1], bootstrap_lower_metric_values.T[0][6:9][::-1], bootstrap_lower_metric_values.T[0][12:15][::-1]])
    bootstrap_upper_metric_values_sparsevi = np.concatenate([bootstrap_upper_metric_values.T[0][3:6][::-1], bootstrap_upper_metric_values.T[0][9:12][::-1], bootstrap_upper_metric_values.T[0][15:][::-1]])
    bootstrap_lower_metric_values_sparsevi = np.concatenate([bootstrap_lower_metric_values.T[0][3:6][::-1], bootstrap_lower_metric_values.T[0][9:12][::-1], bootstrap_lower_metric_values.T[0][15:][::-1]])
    y = np.arange(groups)
    bar_height = 0.2
    set_figure_params(2.6, 2)
    fig, ax = plt.subplots()
    ax.barh(y + bar_height / 2, values_sparsevi, xerr=[bootstrap_upper_metric_values_multivi - values_multivi, values_multivi - bootstrap_lower_metric_values_multivi], height=bar_height, label='SparseVI', color='#70acc0')
    ax.barh(y - bar_height / 2, values_multivi, xerr=[bootstrap_upper_metric_values_multivi - values_multivi, values_multivi - bootstrap_lower_metric_values_multivi], height=bar_height, label='MultiVI', color='#c12075')
    CITE_labels = ['Both modalities', 'RNA only', 'Protein only'][::-1]
    Multiome_labels = ['Both modalities', 'RNA only', 'ATAC only'][::-1]
    labels = Multiome_labels + CITE_labels + CITE_labels
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel(metric_label)
    ax.axhline(y=2.5, color='black', linestyle='--', linewidth=1)
    ax.axhline(y=5.5, color='black', linestyle='--', linewidth=1)
    ax.text(-0.4, 7, 'NeurIPS\n2021CITE\n-> Hao2021\nCITE', va='center', ha='center', verticalalignment='center', rotation=90, fontsize=5.5)
    ax.text(-0.4, 4, 'NeurIPS\n2021\nCITE', va='center', ha='center', verticalalignment='center', rotation=90, fontsize=5.5)
    ax.text(-0.4, 1, 'NeurIPS\n2021\nMultiome', va='center', ha='center', verticalalignment='center', rotation=90, fontsize=5.5)
    plt.subplots_adjust(left=0.5)
    plt.tight_layout()
    if save_name is None:
        plt.show()
    else:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')


In [ ]:
mode = "subset"


In [ ]:
for metric, metric_label in zip(metrics_aggregated.keys(), metrics_labels):
    metric_aggregated_values = bootstrap_mean_metrics_aggregated[metric]
    metric_filtered_aggregated_values = bootstrap_mean_metrics_filtered_aggregated[metric]
    bootstrap_ci_lower_metrics_aggregated_values = bootstrap_ci_lower_metrics_aggregated[metric]
    bootstrap_ci_higher_metrics_aggregated_values = bootstrap_ci_higher_metrics_aggregated[metric]
    bootstrap_ci_lower_metrics_filtered_aggregated_values = (
        bootstrap_ci_lower_metrics_filtered_aggregated[metric]
    )
    bootstrap_ci_higher_metrics_filtered_aggregated_values = (
        bootstrap_ci_higher_metrics_filtered_aggregated[metric]
    )
    metric_values = np.stack(
        [
            np.array(metric_aggregated_values),
            np.array(metric_aggregated_values) - np.array(metric_aggregated_values),
        ],
        axis=1,
    )
    metric_filtered_values = np.stack(
        [
            np.array(metric_filtered_aggregated_values),
            np.array(metric_filtered_aggregated_values)
            - np.array(metric_filtered_aggregated_values),
        ],
        axis=1,
    )
    bootstrap_upper_metric_values = np.stack(
        [
            np.array(bootstrap_ci_higher_metrics_aggregated_values),
            np.zeros_like(np.array(bootstrap_ci_higher_metrics_aggregated_values)),
        ],
        axis=1,
    )
    bootstrap_lower_metric_values = np.stack(
        [
            np.array(bootstrap_ci_lower_metrics_aggregated_values),
            np.zeros_like(np.array(bootstrap_ci_lower_metrics_aggregated_values)),
        ],
        axis=1,
    )
    bootstrap_filtered_upper_metric_values = np.stack(
        [
            np.array(bootstrap_ci_higher_metrics_filtered_aggregated_values),
            np.zeros_like(np.array(bootstrap_ci_higher_metrics_filtered_aggregated_values)),
        ],
        axis=1,
    )
    bootstrap_filtered_lower_metric_values = np.stack(
        [
            np.array(bootstrap_ci_lower_metrics_filtered_aggregated_values),
            np.zeros_like(np.array(bootstrap_ci_lower_metrics_filtered_aggregated_values)),
        ],
        axis=1,
    )

    plot_metric_vertical_v2(
        mode,
        metric_values,
        metric_label,
        bootstrap_upper_metric_values,
        bootstrap_lower_metric_values,
        f"label_transfer_vertical_only_not_filtered_100_bootstrap_{metric_label}_{mode}",
    )


In [ ]:
import os

os.getcwd()
